In [ ]:
import collections
from sage.all import CartanType, RootSystem, QQ, ZZ, PuiseuxSeriesRing, crystals, O


def affine_char_list(cartan_type, level, max_ord):
    """
    Computes the specialized characters of an affine Lie algebra at a given level
    as a list of Puiseux series up to O(q^max_ord) relative to the leading term.
    """
    # 1. Normalize and resolve the Cartan types
    ct = CartanType(cartan_type)
    if ct.is_affine():
        ct_affine = ct
        ct_finite = ct.classical()
    else:
        ct_finite = ct
        ct_affine = ct.affine()
        
    # 2. Initialize the Puiseux Series Ring over Integers
    R = PuiseuxSeriesRing(ZZ, 'q')
    q = R.gen()
    
    # 3. Compute central charge constants using the core root system
    h_check = ct_finite.dual_coxeter_number()
    # dim(g) = rank(g) + number of roots (completely bypasses LieAlgebra factory errors)
    dim_g = ct_finite.rank() + len(list(RootSystem(ct_finite).root_lattice().roots()))
    c = QQ(level * dim_g) / QQ(level + h_check)
    
    # 4. Set up ambient space to compute precise inner products for h
    L_finite = RootSystem(ct_finite).ambient_space()
    fw_finite = L_finite.fundamental_weights()
    rho_finite = L_finite.rho()
    
    # 5. Find all dominant highest weights of the given level using the dual null root
    indices = list(ct_affine.index_set())
    
    # Comarks of ct_affine equal the marks of its dual affine type
    # We find them by looking at the null root (delta) of the dual root lattice
    Q_dual = RootSystem(ct_affine.dual()).root_lattice()
    delta_dual = Q_dual.null_root()
    colabels = {i: delta_dual.coefficient(i) for i in indices}

    dominant_weights_coeffs = []
    
    def find_weights(idx, current_sum, current_dict):
        if idx == len(indices):
            if current_sum == level:
                dominant_weights_coeffs.append(current_dict.copy())
            return
        i = indices[idx]
        colab = colabels[i]
        max_val = (level - current_sum) // colab
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * colab, current_dict)
            del current_dict[i]
            
    find_weights(0, 0, {})
    
    # 6. Set up the affine weight lattice to build crystals
    P_affine = RootSystem(ct_affine).weight_lattice()
    Lambda_affine = P_affine.fundamental_weights()
    
    characters = []
    
    # 7. Traverse the crystal for each valid highest weight
    finite_indices = list(ct_finite.index_set())
    
    for coeffs in dominant_weights_coeffs:
        # Accumulate the finite weight projection manually to avoid integer-zero addition errors
        bar_Lambda = fw_finite[finite_indices[0]] * coeffs.get(finite_indices[0], 0)
        for i in finite_indices[1:]:
            bar_Lambda += fw_finite[i] * coeffs.get(i, 0)
            
        # Conformal weight h = (Lambda, Lambda + 2*rho) / (2 * (level + h^V))
        inner_prod = bar_Lambda.inner_product(bar_Lambda + 2 * rho_finite)
        h = QQ(inner_prod) / QQ(2 * (level + h_check))
        
        # Leading modular exponent: h - c/24
        leading_exponent = h - c / QQ(24)
        
        # Accumulate the affine weight vector manually
        weight = Lambda_affine[indices[0]] * coeffs[indices[0]]
        for i in indices[1:]:
            weight += Lambda_affine[i] * coeffs[i]
            
        # Instantiate the highest weight crystal module
        B = crystals.HighestWeight(weight)
        b0 = B.highest_weight_vector()
        
        # Graded dimension tracker via BFS
        counts = [0] * (max_ord + 1)
        queue = collections.deque([(b0, 0)])
        visited = {b0}
        
        while queue:
            b, c0 = queue.popleft()
            counts[c0] += 1
            
            for i in indices:
                fb = b.f(i)
                if fb is not None:
                    # Every application of f_0 lowers the weight by alpha_0, increasing the energy grading by 1
                    new_c0 = c0 + (1 if i == 0 else 0)
                    if new_c0 <= max_ord and fb not in visited:
                        visited.add(fb)
                        queue.append((fb, new_c0))
        
        # 8. Assemble the Puiseux series
        series_sum = sum(counts[depth] * q**depth for depth in range(max_ord + 1))
        char_series = q**leading_exponent * series_sum + O(q**(leading_exponent + max_ord))
        characters.append(char_series)
        
    return characters

In [ ]:
from collections import defaultdict
def affine_char_list(cartan_type, level, max_ord=15):
    '''
    Input: a cartan type (e.g. "A4"), a level, and the maximum order to compute the characters to
    Output: the list of normalised specialised characters of the affine lie algebra
    '''
    # Get the Cartan type
    ct = CartanType(cartan_type)
    if not ct.is_affine():
        ct = ct.affine()
    # Indices
    index = list( ct.index_set() )
    # Affine root system
    WL = RootSystem(ct).weight_lattice(extended=True)
    # Simple roots
    simple_roots = WL.simple_roots()
    # Simple coroots
    simple_coroots = WL.simple_coroots()
    # List of fundamental weights
    Lambda = WL.fundamental_weights()
    # The rank
    rnk = len(index)-1
    # The null root delta
    delta = WL.null_root()
    # The (classical) Cartan matrix
    cartan_mat = CartanMatrix(ct.classical())
    inv_cartan_mat = cartan_mat.inverse()
    # Highest root
    highest_root = WL.classical().highest_root()
    # The quadratic form 
    quadF = matrix(QQ, [[inv_cartan_mat[i,j]*simple_roots[i+1].norm_squared()/highest_root.norm_squared() for i in range(rnk)] for j in range(rnk)] )
    max_F = max(quadF.coefficients())
    
    # Now we find the (classical) weight, root, and coroot lattices as sagemath free module objects over the integers
    weight_lattice = span([vector(ZZ,rnk,{i:1}) for i in range(rnk)],ZZ)
    root_lattice = span(cartan_mat.columns(),ZZ)
    coroot_lattice = span([cartan_mat.columns()[i]*highest_root.norm_squared()/simple_roots[i+1].norm_squared() for i in range(rnk)],ZZ)

    # The power series ring 
    P = PuiseuxSeriesRing(ZZ,'q')
    q = P.gen()

    # Define the generalised theta functions
    def Theta(lamb):
        r'''
        Generalised theta function
        Input: a weight
        Output: the specialised generalised theta function associated to that weight
        '''
        vec_lamb = lamb.to_classical().to_vector()
        dictseries = defaultdict(int)
        qF_gcd = 1/gcd(quadF.coefficients())
        e = 2*level*qF_gcd
        for cr in coroot_lattice:
            vec = level*cr+vec_lamb
            exponent = qF_gcd*vec*quadF*vec
            if exponent <= e*max_ord:
                dictseries[exponent] += 1
            elif exponent > 2*e*max_F*max_ord:
                break
        
        return P(dictseries,e=e).add_bigoh(max_ord)
  

    # Get the affine Weyl group and isolate the classical generators
    affine_weyl = WL.weyl_group()
    classical_nodes = [i for i in index if i != 0]
    weyl_gen = [affine_weyl.simple_reflection(i) for i in classical_nodes]

    # Comarks of ct_affine equal the marks of its dual affine type
    # We find them by looking at the null root (delta) of the dual root lattice
    Q_dual = RootSystem(ct.dual()).root_lattice()
    delta_dual = Q_dual.null_root()
    comarks = {i: delta_dual.coefficient(i) for i in index}

    # Now we find all the integrable weights
    int_weights_coeffs = []
    def find_weights(idx, current_sum, current_dict):
        if idx == rnk+1:
            if current_sum == level:
                int_weights_coeffs.append(current_dict.copy())
            return
        i = index[idx]
        max_val = (level - current_sum) // comarks[i] 
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * comarks[i], current_dict)
            del current_dict[i]
            
    find_weights(0, 0, {})
    # Number of integrable highest weight representations
    num_reps = len(int_weights_coeffs)
    # The integrable weights
    int_weights = [ sum(Lambda[i]*int_weights_coeffs[j][i] for i in index) for j in range(num_reps)] 
    def are_equivalent(w1, w2):
        '''Function to determine if two weights are related via shift by a coroot'''

        c1 = vector(QQ, [w1.monomial_coefficients().get(i+1,0) for i in range(rnk)] )
        c2 = vector(QQ, [w2.monomial_coefficients().get(i+1,0) for i in range(rnk)] )
        return (c1-c2)/level in coroot_lattice #and w1.monomial_coefficients().get(0,0)==w2.monomial_coefficients().get(0,0)

    # Now loop over these weights and calculate the character for each
    char_list = []
    for iwght in int_weights:
        # The corresponding integrable representation
        int_rep = IntegrableRepresentation(iwght)
        # First let's calculate the set of maximal dominant weights
        md_weights = int_rep.dominant_maximal_weights()
        # Now we find a full set of representatives up to translations by the coroot lattice
        full_maximal_representatives = list( md_weights )
        queue = list( md_weights )
        while queue:
            current = queue.pop(0)
            for g in weyl_gen:
                nxt = g.action(current)
                # Only keep nxt if it represents a brand new class modulo the coroots
                if not any(are_equivalent(nxt, r) for r in full_maximal_representatives):
                    full_maximal_representatives.append(nxt)
                    queue.append(nxt)
        
        # Now compute the character by summing over these weights
        char = P(0).add_bigoh(max_ord)
        for wght in full_maximal_representatives:
            char += q**(int_rep.modular_characteristic(wght))*P( int_rep.string(wght, depth=max_ord) )*Theta(wght)
        # Update the list of characters
        char_list.append(char)
        
    return char_list